# Imputation Technique 1: Dropping High-Missing Columns

**Dataset:** `Loan_Default.csv`

**When to use:** When a column has so many missing values that imputing them would introduce too much noise or distortion.

**Key concept:** If a column is missing more than a threshold (the lab uses **> 10,000** missing values), it's more statistically honest to drop the whole column rather than fabricate too much data.

---


### Step 1: Setup & Data Loading


In [ ]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

# Load data
df = pd.read_csv('../../data/raw/Loan_Default.csv')
df.drop(['ID', 'year'], axis=1, inplace=True)

# Pre-encode features so we have a clean numerical/categorical baseline
categorical_features = df.select_dtypes(include=['object']).columns.tolist()
Ordinal_features = ['age']
Nominal_features = categorical_features.copy()
Nominal_features.remove('age')

enc = OrdinalEncoder()
df[Ordinal_features] = enc.fit_transform(df[Ordinal_features])

df_prep = df.copy()
for c in Nominal_features:
    df_prep[c + '_freq'] = df_prep[c].map(df_prep.groupby(c).size() / df_prep.shape[0])
    indexer = pd.factorize(df_prep[c], sort=True)[1]
    df_prep[c] = indexer.get_indexer(df_prep[c])
df_prep = df_prep.drop(Nominal_features, axis=1)

print(f'Starting shape: {df_prep.shape}')

### Step 2: Identify Missing Values Per Column

Before deciding what to drop, we inspect the scale of missing data.


In [ ]:
missing = df_prep.isna().sum().sort_values(ascending=False)
print(missing[missing > 0])

### Step 3: Drop High-Missing Columns

The lab defines "high missing" as > 10,000 missing values. These columns are dropped entirely.


In [ ]:
high_missing_cols = [
    'rate_of_interest',
    'Interest_rate_spread',
    'Upfront_charges',
    'property_value',
    'LTV',
    'dtir1'
]

df_drop = df_prep.drop(high_missing_cols, axis=1)

print(f'Shape after dropping high-missing columns: {df_drop.shape}')
df_drop.head()

### Step 4: Verify Remaining Missing Values

After dropping, we check what missing data remains. These will be handled by other strategies like Mode or Mean imputation.


In [ ]:
remaining_missing = df_drop.isna().sum()
print(remaining_missing[remaining_missing > 0])